# Iniciando o Spark

In [1]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_bureau") \
    .getOrCreate()

# Importando bibliotecas

In [ ]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [ ]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

# Buckets e nomes de saída
bucket_base = "base_bureau"
bucket_trusted = f"s3://hackathon_2025/{PROCESS_DATE}/0003_trusted/"
bucket_raw = f"s3://hackathon_2025/{PROCESS_DATE}/0002_raw/"
bucket_control = f"s3://hackathon_2025/{PROCESS_DATE}/0005_control/"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_trusted:", bucket_trusted)
print("bucket_raw:", bucket_raw)
print("bucket_control:", bucket_control)


# Leitura dos dados na camada Raw

In [ ]:
path_raw = os.path.join(bucket_raw, bucket_base)
df_raw = spark.read.parquet(path_raw)
df_raw.createOrReplaceTempView("raw_base_bureau")

print(log(), "Registros na Raw:", df_raw.count())
df_raw.show(5, truncate=False)


# Processamento tipagem para camada Trusted

In [ ]:
df_trusted = spark.sql(f"""
    SELECT
        '{dthproc}' AS ts_proc,
        '{dthproc}' AS ts_proc_partition,
        -- SAFRA convertida para DATE (primeiro dia do mês)
        CAST(CONCAT(SUBSTRING(SAFRA, 1, 4), '-', SUBSTRING(SAFRA, 5, 2), '-01') AS DATE) AS SAFRA,

        -- Ano e Mês extraídos da SAFRA
        CAST(SUBSTRING(SAFRA, 1, 4) AS INT) AS Ano,
        CAST(SUBSTRING(SAFRA, 5, 2) AS INT) AS Mes,

        CAST(FLAG_INSTALACAO AS BOOLEAN) AS IsInstallation,
        CAST(PROD AS STRING) AS ProductDescription,
        CAST(flag_mig2 AS STRING) AS ProductMigration,
        CAST(SCORE_01 AS FLOAT) AS Score01,
        CAST(SCORE_02 AS FLOAT) AS Score02,
        CAST(FPD AS INT) AS FDP,
        CAST(NUM_CPF AS STRING) AS NUM_CPF
    FROM raw_base_bureau
""")

df_trusted.createOrReplaceTempView("lake_base_bureau")
#df_trusted.cache()

print(log(), "Registros Trusted:", df_trusted.count())
#df_trusted.printSchema()
df_trusted.show(5, truncate=False)

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_trusted.write \
    .partitionBy("SAFRA","ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

# Controle de carga

In [ ]:
controle = spark.sql(f"""
    SELECT
        '{output_trusted}' AS name_file,
        ts_proc,
        ts_proc_partition,
        COUNT(*) AS qtd_registros
    FROM lake_base_bureau
    GROUP BY 1,2,3
""")

controle.createOrReplaceTempView("controle")
#controle.cache()

print(log(), "Registros controle:", controle.count())
controle.show(truncate=False)

# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f"tb_0002_controle_processamento_{bucket_base}_trusted")
print("Control path:", path_control)

controle.write \
    .mode("append") \
    .option("compression", "snappy") \
    .parquet(path_control)